In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.silver")

In [ ]:
from pyspark.sql import functions as F

In [ ]:
# Use summary for sentiment when available — more context than title alone.
# Fall back to title if summary is empty/null.
# ai_analyze_sentiment() is a Databricks built-in Foundation Model function.
scored = (
    spark.table("stocks.bronze.news")
    .withColumn(
        "sentiment_input",
        F.when(
            F.col("summary").isNotNull() & (F.length(F.trim(F.col("summary"))) > 0),
            F.col("summary")
        ).otherwise(F.col("title"))
    )
    .withColumn("sentiment", F.expr("ai_analyze_sentiment(sentiment_input)"))
    .drop("sentiment_input")
)

In [ ]:
# Aggregate to one row per ticker per day
daily = (
    scored.groupBy("ticker", "date")
    .agg(
        F.count("*").alias("article_count"),
        F.sum(F.when(F.col("sentiment") == "positive",  1).otherwise(0)).alias("positive_count"),
        F.sum(F.when(F.col("sentiment") == "negative",  1).otherwise(0)).alias("negative_count"),
        F.sum(F.when(F.col("sentiment") == "neutral",   1).otherwise(0)).alias("neutral_count"),
        # Normalised score: -1 (all negative) to +1 (all positive)
        (
            F.sum(
                F.when(F.col("sentiment") == "positive",  1)
                 .when(F.col("sentiment") == "negative", -1)
                 .otherwise(0)
            ) / F.count("*")
        ).alias("sentiment_score"),
    )
)

daily.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("stocks.silver.news_sentiment")

print(f"stocks.silver.news_sentiment: {spark.table('stocks.silver.news_sentiment').count()} rows")
spark.table("stocks.silver.news_sentiment").orderBy(F.desc("date")).show(10, truncate=False)